# Real world tables

Toy datasets come all numeric and fully informed, real ones rarely do. In this example we work with a customer table where we want to predict who will leave (churn). It has everything we usually find: categories, a code duplicating one of them, free text job titles, dates, a column that never changes and missing values.

Falcondale relies on [skrub](https://skrub-data.org/) for this, available with `pip install "falcondale[tabular]"`.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 500

plan = rng.choice(["basic", "plus", "premium"], n, p=[0.5, 0.3, 0.2])
tenure = rng.integers(1, 72, n).astype(float)
fee = np.select([plan == "basic", plan == "plus"], [20, 45], 80) + rng.normal(0, 5, n)
calls = rng.poisson(1.5, n).astype(float)
# Customers tend to leave in their first year and again around their sixth one
risk = 1.5 * np.cos(np.pi * tenure / 36) + 0.6 * calls - 0.03 * (fee - 45) + rng.normal(0, 0.6, n)

customers = pd.DataFrame(
    {
        "plan": plan,
        "plan_code": pd.Series(plan).map({"basic": "B", "plus": "P", "premium": "X"}),
        "city": rng.choice(["Bilbao", "Donostia", "Gasteiz", "Iruña", None], n),
        "tenure_months": tenure,
        "monthly_fee": fee,
        "support_calls": calls,
        "signup_date": (pd.Timestamp("2026-10-01") - pd.to_timedelta(tenure * 30, unit="D")).strftime("%Y-%m-%d"),
        "country": "ES",
        "job": rng.choice([f"role {i}" for i in range(40)], n),
        "churn": (risk > 1.2).astype(int),
    }
)
customers.loc[rng.choice(n, n // 4, replace=False), "monthly_fee"] = np.nan
customers.loc[rng.choice(n, n // 16, replace=False), "support_calls"] = np.nan
customers.head()

,plan,plan_code,city,tenure_months,monthly_fee,support_calls,signup_date,country,job,churn
0,plus,P,Iruña,40.0,41.572790,NaN,2023-06-19,ES,role 35,0
1,basic,B,NaN,6.0,18.097084,3.0,2026-04-04,ES,role 37,1
2,basic,B,Bilbao,14.0,20.230465,NaN,2025-08-07,ES,role 36,1
3,basic,B,Iruña,63.0,NaN,0.0,2021-07-29,ES,role 26,1
4,premium,X,NaN,32.0,78.611732,1.0,2024-02-14,ES,role 27,0


Profiling the dataset already tells us there is some work to do.

In [2]:
from falcondale import Project

myproject = Project(customers, target="churn")
myproject.profile_dataset()

Number of samples on the dataset: 500
Number of columns on the dataset: 10
churn has been selected as target variable for supervised learning tasks.
Target class balance: 0: 59.2%, 1: 40.8%
Ratio of cells with missing values: 4.82%
Some features are not numeric, preprocess() will encode them with skrub: ['plan', 'plan_code', 'city', 'signup_date', 'country', 'job']
Constant columns carry no information and could be dropped: ['country']


{'samples': 500,
 'columns': 10,
 'missing_ratio': 0.0482,
 'duplicated_rows': 0,
 'non_numeric_columns': ['plan',
  'plan_code',
  'city',
  'signup_date',
  'country',
  'job'],
 'constant_columns': ['country'],
 'target_balance': {0: 0.592, 1: 0.408}}

`table_report()` gives a closer, interactive look at every column, the missing values and the associations between columns. We do not render it here to keep this page light, run it in your notebook or call `.open()` on it to see it in the browser.

```python
myproject.table_report()
```

## Encoding

`preprocess()` notices the non numeric columns and encodes each one according to its kind: categories are one hot encoded, job titles are compressed into a few components, dates are split into numeric parts and `country` is dropped. Missing values are then filled with the median.

In [3]:
myproject.preprocess()
features = myproject.show_features()
print(
    f"{customers.shape[1] - 1} columns became {features.shape[1]} features, {features.isna().sum().sum()} missing values left"
)
features.head()

9 columns became 22 features, 0 missing values left


,plan_basic,plan_plus,plan_premium,plan_code_B,plan_code_P,plan_code_X,city_Bilbao,city_Donostia,city_Gasteiz,city_Iruña,...,monthly_fee,support_calls,signup_date_year,signup_date_month,signup_date_day,signup_date_total_seconds,job_0,job_1,job_2,job_3
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.427275,0.2,0.500000,0.454545,0.600000,0.442857,0.213343,0.058322,0.441524,0.425222
1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.160217,0.6,1.000000,0.272727,0.100000,0.928571,0.162617,0.074375,0.473555,0.455265
2,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.184486,0.2,0.833333,0.636364,0.200000,0.814286,0.213347,0.071793,0.540825,0.435519
3,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.399525,0.0,0.166667,0.545455,0.933333,0.114286,0.188041,0.080758,0.453640,0.423312
4,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.848629,0.2,0.666667,0.090909,0.433333,0.557143,0.213336,0.072916,0.486154,0.410301


## Assessment

With 22 features we are well beyond what one qubit per feature allows on a laptop. The assessment diagnoses the raw table as well, so the report shows how many features each column became, how related it is to the target and which steps could help.

In [4]:
myproject.assess(n_qubits=12)

╭─ QML suitability ────────────────────────────────────────────────────────────╮
│ 84.8 / 100   HIGH  Quantum models are worth a try                            │
│ 500 of 500 samples, 22 of 22 features, 2 classes, budget of 12 qubits        │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Sub-scores ─────────────────────────────────────────────────────────────────╮
│ Fourier approx.     ━━━━━━━━━━━━━━━━━━╺━━━     82  average R² 0.685          │
│ Trainability        ━━━━━━━━━━━━━━━━━━━╺━━     88  12.1% of frequencies      │
│                                                    unreachable (ternary      │
│                                                    init)                     │
│ Circuit depth       ━━━━━━━━━━━━━╺━━━━━━━━     60  fixed frequency depth ≈   │
│                                                    864                       │
│ Classical hardness  ━━━━━━━━━━━━━━━━━━━━━━    100  logistic bal. acc. 0.695, │
│                                                    tree gap +0.076           │
│ Class balance       ━━━━━━━━━━━━━━━━━━╺━━━     83  ratio 0.689 (mild)        │
│ Dimensionality      ━━━━━━━━━━━━━━━━━━━━━━    100  ≥ 5 qubits needed         │
╰──────────────────────────────────────────────────────────────────────────────╯
Raw table: 9 columns become 22 features                                         
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Column          ┃ Kind      ┃  Missing ┃  Unique ┃  Features ┃ Target assoc. ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ plan            │ category  │       0% │       3 │         3 │          0.40 │
│ plan_code       │ category  │       0% │       3 │         3 │          0.40 │
│ city            │ category  │      17% │       4 │         5 │          0.10 │
│ tenure_months   │ numeric   │       0% │      71 │         1 │          0.63 │
│ monthly_fee     │ numeric   │      25% │     375 │         1 │          0.37 │
│ support_calls   │ numeric   │       6% │       6 │         1 │          0.26 │
│ signup_date     │ date      │       0% │      71 │         4 │          0.29 │
│ country         │ dropped   │       0% │       1 │         0 │          0.00 │
│ job             │ text      │       0% │      40 │         4 │          0.15 │
└─────────────────┴───────────┴──────────┴─────────┴───────────┴───────────────┘
Classical baseline (3 fold, out of fold)                                        
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ Model                 ┃  Bal. accuracy ┃  Precision ┃ Recall ┃    F1 ┃   AUC ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ Logistic regression   │          0.695 │      0.581 │  0.775 │ 0.664 │ 0.757 │
│ Decision tree         │          0.771 │      0.744 │  0.711 │ 0.727 │ 0.785 │
└───────────────────────┴────────────────┴────────────┴────────┴───────┴───────┘
Spectrum                                  
┌─────────────────────────────┬──────────┐
│ Features showing signal     │ 11 of 22 │
│ Highest frequency ω         │        8 │
│ Significant frequencies K   │       16 │
│ Unreachable, zero init      │    28.8% │
│ Unreachable, ternary init   │    12.1% │
├─────────────────────────────┼──────────┤
│ RBF kernel effective rank   │    13.03 │
│ RBF kernel spectral gap     │    8.422 │
│ RBF kernel condition number │ 7.76e+08 │
└─────────────────────────────┴──────────┘
Most oscillating features                                                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Feature                    ┃    ω ┃    K ┃         R² ┃          Unreachable ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ job_2                      │    8 │    1 │      0.318 │                  1/1 │
│ monthly_fee                │    4 │    3 │      0.689 │                  1/3 │
│

The raw table tells a clear story. `plan` and `plan_code` carry the same information, `city` and `job` are only weakly related to churn (Cramér's V of 0.10 and 0.15) while adding 9 features, and the date only repeats what `tenure_months` says in 4 features. Let's keep the relevant columns and assess again.

In [5]:
compact = Project(customers[["plan", "tenure_months", "monthly_fee", "support_calls", "churn"]], target="churn")
compact.preprocess()
compact.assess(n_qubits=12)

╭─ QML suitability ────────────────────────────────────────────────────────────╮
│ 84.7 / 100   HIGH  Quantum models are worth a try                            │
│ 500 of 500 samples, 6 of 6 features, 2 classes, budget of 12 qubits          │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Sub-scores ─────────────────────────────────────────────────────────────────╮
│ Fourier approx.     ━━━━━━━━━━━━━━━━╸━━━━━     76  average R² 0.636          │
│ Trainability        ━━━━━━━━━━━━━━━━━━━━╸━     93  6.7% of frequencies       │
│                                                    unreachable (ternary      │
│                                                    init)                     │
│ Circuit depth       ━━━━━━━━━━━━━╺━━━━━━━━     60  fixed frequency depth ≈   │
│                                                    416                       │
│ Classical hardness  ━━━━━━━━━━━━━━━━━━━━━━    100  logistic bal. acc. 0.677, │
│                                                    tree gap +0.145           │
│ Class balance       ━━━━━━━━━━━━━━━━━━╺━━━     83  ratio 0.689 (mild)        │
│ Dimensionality      ━━━━━━━━━━━━━━━━━━━━━━    100  ≥ 4 qubits needed         │
╰──────────────────────────────────────────────────────────────────────────────╯
Raw table: 4 columns become 6 features                                          
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Column          ┃ Kind      ┃  Missing ┃  Unique ┃  Features ┃ Target assoc. ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ plan            │ category  │       0% │       3 │         3 │          0.40 │
│ tenure_months   │ numeric   │       0% │      71 │         1 │          0.63 │
│ monthly_fee     │ numeric   │      25% │     375 │         1 │          0.37 │
│ support_calls   │ numeric   │       6% │       6 │         1 │          0.26 │
└─────────────────┴───────────┴──────────┴─────────┴───────────┴───────────────┘
Classical baseline (3 fold, out of fold)                                        
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ Model                 ┃  Bal. accuracy ┃  Precision ┃ Recall ┃    F1 ┃   AUC ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ Logistic regression   │          0.677 │      0.555 │  0.794 │ 0.653 │ 0.744 │
│ Decision tree         │          0.823 │      0.796 │  0.784 │ 0.790 │ 0.832 │
└───────────────────────┴────────────────┴────────────┴────────┴───────┴───────┘
Spectrum                                 
┌─────────────────────────────┬─────────┐
│ Features showing signal     │  5 of 6 │
│ Highest frequency ω         │       4 │
│ Significant frequencies K   │       8 │
│ Unreachable, zero init      │   23.3% │
│ Unreachable, ternary init   │    6.7% │
├─────────────────────────────┼─────────┤
│ RBF kernel effective rank   │    3.97 │
│ RBF kernel spectral gap     │  15.976 │
│ RBF kernel condition number │ 5.1e+13 │
└─────────────────────────────┴─────────┘
Most oscillating features                                                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Feature                    ┃    ω ┃    K ┃         R² ┃          Unreachable ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ monthly_fee                │    4 │    3 │      0.689 │                  1/3 │
│ plan_premium               │    2 │    2 │      0.721 │                  0/2 │
│ plan_basic                 │    1 │    1 │      0.596 │                  0/1 │
│ tenure_months              │    1 │    1 │      0.875 │                  0/1 │
│ support_calls              │    1 │    1 │      0.298 │                  0/1 │
│ plan_plus                  │    0 │    0 │      0.000 │                  0/0 │
└────────────────────────────┴──────┴──────┴────────────┴──────────────────────┘
Recommended m

Six features fit the budget of a quantum kernel now. `evaluate("auto")` trains the recommended model, and we compare it with the classical baseline.

In [6]:
model = compact.evaluate("auto")
baseline = compact.evaluate("svc")

print(f"{model.type}: {model.metric('auc'):.3f} AUC, svc: {baseline.metric('auc'):.3f} AUC")

Assessment recommends qsvc (backend=qiskit): 93% of the frequencies within reach of a repeated encoding


qsvc: 0.804 AUC, svc: 0.923 AUC


The quantum kernel does a decent job but the classical SVC wins on this dataset. That is the reason Falcondale always offers the baseline: the assessment points to the models worth trying first, the comparison has the final word.

The encoder is part of the model, so new customers can be scored straight from their raw rows, missing values included.

In [7]:
new_customers = pd.DataFrame(
    {
        "plan": ["basic", "premium", "plus"],
        "tenure_months": [3.0, 40.0, 70.0],
        "monthly_fee": [21.0, None, 47.0],
        "support_calls": [4.0, 0.0, None],
    }
)
model.predict_proba(new_customers)

array([0.5071791 , 0.16502873, 0.61114795])